# Module 00 — Prerequisites & Python Refresher

**Prerequisites:** Basic Python (variables, loops, functions, classes)
**Time:** ~75 minutes
**Part of:** PyTorch From Zero to Practitioner

## Learning Objectives

By the end of this notebook you will be able to:

- Recognize the small set of Python features PyTorch code leans on heavily: classes and `__init__`/`super()`, `*args`/`**kwargs`, list comprehensions, context managers (`with`), and iterators.
- Explain why PyTorch is built around **classes** rather than plain functions.
- Read a "for batch in loader:" style loop and know exactly what it's iterating over.
- Use `*args`/`**kwargs`, lambda functions, and decorators confidently — all three appear constantly in PyTorch code.
- Understand, at a high level, what "array-based numerical computing" (NumPy-style thinking) means, since PyTorch tensors extend this idea.

## Why This Matters

You don't need to be a Python expert to learn PyTorch, but PyTorch's API assumes fluency with a handful of specific patterns. If those patterns are shaky, every PyTorch notebook you read afterward will feel harder than it needs to be. This module is a fast, targeted refresher — not a full Python course — aimed exactly at what you'll see starting in the very next notebook.

If everything below already looks familiar, skim it in five minutes and move on. If any section feels new, slow down here — it will pay off immediately.


## 1. Classes, `__init__`, and `super()`

Almost every model you build in PyTorch is a **class** that inherits from `torch.nn.Module`. So before touching PyTorch, let's make sure the underlying Python mechanics are second nature.


In [ ]:
class Animal:
    def __init__(self, name, sound):
        # __init__ runs automatically when you create an object: Animal("Dog", "Woof")
        self.name = name
        self.sound = sound

    def speak(self):
        return f"{self.name} says {self.sound}"


class Dog(Animal):
    def __init__(self, name):
        # super().__init__(...) calls the PARENT class's __init__.
        # This lets Dog reuse Animal's setup logic instead of duplicating it.
        super().__init__(name, sound="Woof")
        self.legs = 4


rex = Dog("Rex")
print(rex.speak())
print(rex.legs)


**What's happening, and why it matters for PyTorch:**

- `Dog` *inherits* from `Animal`, meaning it gets all of `Animal`'s behavior for free, then adds its own (`self.legs`).
- `super().__init__(...)` is how a child class asks its parent to do its own setup first.

This is *exactly* the pattern you will type dozens of times in PyTorch:

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()   # let nn.Module do its internal setup
        self.layer = nn.Linear(10, 1)   # then add your own stuff
```

If `super().__init__()` looked unfamiliar above, re-read this section before continuing — it is the single most important piece of "plain Python" background for PyTorch.


### 🔮 Predict before you run

What do you think happens if we forget `super().__init__(name, sound="Woof")` inside `Dog.__init__`, and instead just write `self.legs = 4`? Will `rex.speak()` still work?

Think about it, then check the cell below.


In [ ]:
class BrokenDog(Animal):
    def __init__(self, name):
        # No super().__init__() call!
        self.legs = 4

broken = BrokenDog("Fido")
try:
    print(broken.speak())
except AttributeError as e:
    print("Error:", e)


**Explanation:** `speak()` is inherited from `Animal`, but it depends on `self.name` and `self.sound`, which are only set inside `Animal.__init__`. Skipping `super().__init__()` means that setup never runs, so those attributes don't exist.

This is precisely why forgetting `super().__init__()` in a PyTorch model produces confusing errors — `nn.Module` sets up internal bookkeeping (like the dictionary that tracks your layers) in its own `__init__`, and skipping it silently breaks things later.


## 2. `*args` and `**kwargs`

These appear in almost every PyTorch function signature. They let a function accept a variable number of positional (`*args`) or keyword (`**kwargs`) arguments without listing every one explicitly.


In [ ]:
# *args: collects extra positional arguments into a tuple
def add_all(*args):
    print(f"Received {len(args)} arguments: {args}")
    return sum(args)

print(add_all(1, 2, 3))         # 3 args
print(add_all(10, 20, 30, 40))  # 4 args — same function handles both


In [ ]:
# **kwargs: collects extra keyword arguments into a dictionary
def describe_config(**kwargs):
    for key, value in kwargs.items():
        print(f"  {key} = {value}")

print("Model config:")
describe_config(learning_rate=0.01, epochs=100, hidden_dim=64)


**Why this matters for PyTorch:** many PyTorch functions accept `**kwargs` to forward configuration options to lower-level functions. For example:

```python
# DataLoader forwards unknown keyword args to the sampler/collate function
DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)
```

You'll also see `*args` in layer constructors like `nn.Sequential(*layers)` — the `*` unpacks a list into separate positional arguments.


In [ ]:
# Unpacking a list with * — you'll use this with nn.Sequential
layers = [1, 2, 3, 4, 5]
print(*layers)          # equivalent to print(1, 2, 3, 4, 5)

# Unpacking a dict with ** — you'll use this to pass config dicts
config = {"lr": 0.01, "weight_decay": 1e-5}
# torch.optim.Adam(model.parameters(), **config)
# is equivalent to:
# torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)
print("Config:", config)


## 3. Iterating and unpacking

PyTorch training loops are built almost entirely out of `for` loops that unpack tuples. Make sure this feels natural:


In [ ]:
pairs = [(1, "a"), (2, "b"), (3, "c")]

for number, letter in pairs:
    print(number, letter)


This exact shape — `for x, y in something:` — is what a PyTorch training loop looks like:

```python
for inputs, targets in train_loader:
    ...
```

`train_loader` yields tuples of `(inputs, targets)` one batch at a time, and the loop unpacks each tuple into two variables, just like `pairs` above.


In [ ]:
# enumerate() — adds an index counter. Very common for progress tracking.
fruits = ["apple", "banana", "cherry"]

for i, fruit in enumerate(fruits):
    print(f"  item {i}: {fruit}")

# In PyTorch:
# for batch_idx, (inputs, targets) in enumerate(train_loader):
#     if batch_idx % 100 == 0:
#         print(f"Processing batch {batch_idx}")


## 4. `with` blocks (context managers)

You'll frequently see code like:

```python
with torch.no_grad():
    ...
```

A `with` block guarantees that some setup happens before the indented code runs, and some cleanup happens after — even if an error occurs inside. A simple built-in example is file handling:


In [ ]:
import tempfile, os

# Create a temporary file for this demo
demo_path = os.path.join(tempfile.gettempdir(), "demo.txt")

with open(demo_path, "w") as f:
    f.write("hello")
# the file is automatically closed here, even if writing had failed

print("File closed:", f.closed)


`torch.no_grad()` works the same way: it turns *off* gradient tracking for everything inside the block, then automatically turns it back on when the block ends. You'll see exactly why that matters in the autograd notebook.


## 5. Lambda functions

A `lambda` is a one-line, anonymous function. You'll encounter them in transforms, custom sorting, and collate functions.


In [ ]:
# A regular function
def double(x):
    return x * 2

# The same thing as a lambda
double_lambda = lambda x: x * 2

print(double(5), double_lambda(5))  # both give 10

# Where lambdas shine: quick throwaway functions
numbers = [3, 1, 4, 1, 5, 9, 2, 6]
print("sorted:", sorted(numbers))
print("sorted by distance from 5:", sorted(numbers, key=lambda x: abs(x - 5)))


**In PyTorch**, you'll see lambdas in transforms:

```python
transforms.Lambda(lambda x: x / 255.0)   # normalize pixel values
```

And in `map`-style operations:

```python
# Apply a function to every parameter
list(map(lambda p: p.shape, model.parameters()))
```


## 6. Generators and `yield`

A generator is a function that produces values one at a time, pausing between each, rather than computing them all at once. This is how PyTorch's `DataLoader` works internally — it doesn't load the entire dataset into memory; it yields one batch at a time.


In [ ]:
def countdown(n):
    """Yields numbers from n down to 1, one at a time."""
    while n > 0:
        yield n      # pauses here, returns n, resumes when the next value is requested
        n -= 1

# Using it in a for loop — each iteration calls the generator for the next value
for num in countdown(5):
    print(num, end=" ")
print()

# This is conceptually what DataLoader does:
# for batch in data_loader:   # each iteration yields the next batch
#     train_on(batch)


## 7. Decorators

A decorator is a function that wraps another function to add behavior. In PyTorch, you'll see `@torch.no_grad()` used as a decorator — it's an alternative to the `with` block form.


In [ ]:
import time

# A decorator that times how long a function takes
def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_add(a, b):
    time.sleep(0.1)  # simulate work
    return a + b

result = slow_add(3, 4)  # prints timing automatically
print("Result:", result)


**In PyTorch**, the two most common decorators you'll see are:

```python
# Decorator form of torch.no_grad() — used on evaluation/inference functions
@torch.no_grad()
def predict(model, x):
    return model(x)

# Equivalent to:
def predict(model, x):
    with torch.no_grad():
        return model(x)
```

Both forms do the same thing. The decorator form is convenient when the *entire* function should run without gradient tracking.


## 8. Type hints

Modern PyTorch code (and PyTorch's own source code) uses type hints extensively. They don't change what the code *does* — Python ignores them at runtime — but they make code vastly easier to read.


In [ ]:
from typing import List, Tuple, Optional, Dict

# Without type hints — you'd need to read the whole function to know what goes in/out
def process(data, scale, bias):
    return [x * scale + bias for x in data]

# With type hints — instantly clear
def process_typed(data: List[float], scale: float, bias: float = 0.0) -> List[float]:
    return [x * scale + bias for x in data]

print(process_typed([1.0, 2.0, 3.0], scale=2.0, bias=1.0))


**Reading PyTorch-style type hints:**

```python
def forward(self, x: Tensor) -> Tensor:
    ...   # takes a Tensor, returns a Tensor

def train(model: nn.Module, loader: DataLoader, epochs: int = 10) -> List[float]:
    ...   # returns a list of floats (loss values)

def predict(x: Tensor, threshold: Optional[float] = None) -> Tuple[Tensor, Tensor]:
    ...   # threshold is optional (can be None); returns two tensors
```

You don't need to write type hints in your own code yet, but being able to *read* them will help you understand PyTorch's documentation and source code.


## 9. Dictionary comprehensions

Just as list comprehensions build lists, dict comprehensions build dictionaries in one line. You'll use these when manipulating model state dicts and tracking metrics.


In [ ]:
# Build a dict from two lists
names = ["layer1.weight", "layer1.bias", "layer2.weight", "layer2.bias"]
sizes = [16, 16, 1, 1]

param_sizes = {name: size for name, size in zip(names, sizes)}
print(param_sizes)

# Filter: only keep entries where size > 1
large_params = {name: size for name, size in param_sizes.items() if size > 1}
print("Large params:", large_params)


**In PyTorch**, you'll see this pattern when filtering or transforming state dicts:

```python
# Save only the weights that belong to the encoder
encoder_state = {k: v for k, v in model.state_dict().items() if "encoder" in k}
```


## 10. NumPy-style thinking (a preview)

If you've used NumPy, PyTorch tensors will feel immediately familiar — same idea, different name. If you haven't, here's the one concept to internalize now:

> Instead of storing numbers in nested Python lists and looping over them one at a time, you store them in a single **array-like object** and operate on the *whole thing at once*.


In [ ]:
# The "slow, no PyTorch/NumPy" way
python_list = [1, 2, 3, 4, 5]
doubled = [x * 2 for x in python_list]   # loop over every element
print(doubled)

# The array-based way (NumPy syntax shown here; PyTorch tensors work identically)
import numpy as np
array = np.array([1, 2, 3, 4, 5])
doubled_array = array * 2   # no explicit loop: every element is doubled at once
print(doubled_array)


Why does this matter? Two reasons that will come up constantly:

1. **Speed** — operating on the whole array at once lets the underlying C/C++ code process everything in a tight, optimized loop instead of slow Python bytecode.
2. **Mental model** — once you stop thinking "loop over each number" and start thinking "operate on the whole array/tensor," reading PyTorch code becomes much easier. A line like `predictions = model(batch_of_images)` computes predictions for an entire batch simultaneously — there's no visible loop, but conceptually every image is processed in parallel.

The next notebook picks up exactly here and introduces `torch.Tensor`, the object at the center of everything in PyTorch.


## Exercises

🟢 **Beginner 1:** Write a class `Vehicle` with `__init__(self, wheels)` and a method `describe()` that returns a string like `"This vehicle has 4 wheels."`. Then write a subclass `Car(Vehicle)` that calls `super().__init__(4)`.

🟢 **Beginner 2:** Given `data = [(1, 10), (2, 20), (3, 30)]`, write a `for` loop that unpacks each tuple into `idx, value` and prints `f"item {idx}: {value}"`.

🟡 **Intermediate 1:** Write a function `build_config(**kwargs)` that accepts any keyword arguments and returns a dictionary with those key-value pairs, plus a default key `"device"` set to `"cpu"` if not provided.

🟡 **Intermediate 2:** Write a generator function `batch_generator(data, batch_size)` that takes a list and yields sublists of length `batch_size`. For example, `list(batch_generator([1,2,3,4,5], 2))` should give `[[1,2], [3,4], [5]]`.

🔴 **Challenge 1:** Using `with open(...)`, write 3 lines of text to a file, then read them back and print each line stripped of its trailing newline.

🔴 **Challenge 2:** Write a decorator called `@call_counter` that counts how many times the decorated function has been called, printing the count each time. (Hint: the wrapper function needs a mutable object — like a list — to track the count, since integers are immutable.)


In [ ]:
# Space for your exercise solutions



### 📝 Exercise Solutions (expand after attempting)


In [ ]:
# --- Solution: Beginner 1 ---
class Vehicle:
    def __init__(self, wheels):
        self.wheels = wheels
    def describe(self):
        return f"This vehicle has {self.wheels} wheels."

class Car(Vehicle):
    def __init__(self):
        super().__init__(4)

print(Car().describe())

# --- Solution: Beginner 2 ---
data = [(1, 10), (2, 20), (3, 30)]
for idx, value in data:
    print(f"item {idx}: {value}")

# --- Solution: Intermediate 1 ---
def build_config(**kwargs):
    config = {"device": "cpu"}   # default
    config.update(kwargs)         # override with user-provided values
    return config

print(build_config(lr=0.01, epochs=50))
print(build_config(lr=0.01, device="cuda"))  # overrides the default

# --- Solution: Intermediate 2 ---
def batch_generator(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

print(list(batch_generator([1, 2, 3, 4, 5], 2)))


## 🧠 Recap Quiz

Before moving on, try to answer these without scrolling back up:

1. What does `super().__init__()` do, and what breaks if you forget it in a PyTorch model?
2. What's the difference between `*args` and `**kwargs`?
3. What does a `with` block guarantee?
4. What's the difference between `@torch.no_grad()` and `with torch.no_grad():`?
5. Why is `model(batch)` faster than looping over individual examples?

**Answers:**

1. It calls the parent class's `__init__`, running its setup code. Without it in `nn.Module`, parameters won't be registered.
2. `*args` collects extra positional args as a tuple; `**kwargs` collects extra keyword args as a dict.
3. Setup runs before the block, cleanup runs after — even if an error occurs.
4. Nothing — they're two syntax forms of the same thing. The decorator form applies to the entire function.
5. Array-based computation processes the whole batch in optimized C/C++ code rather than slow Python loops.


## Common Mistakes

- **Forgetting `super().__init__()`** in a subclass — leads to missing attributes and confusing `AttributeError`s later. In PyTorch specifically, forgetting this in an `nn.Module` subclass causes errors about parameters not being registered.
- **Confusing `self` with the class name** — `self` refers to *this particular instance*, not the class itself. Every method that needs access to an object's own data takes `self` as its first parameter.
- **Confusing `*args` with `**kwargs`** — `*args` is for positional arguments (a tuple), `**kwargs` is for keyword arguments (a dict). Mixing them up leads to `TypeError`s.
- **Assuming array operations loop invisibly, one element at a time in Python** — they don't. That's the whole point: the looping happens in fast, compiled code, not in Python.

## Mental Model

Think of a class as a *blueprint* and `__init__` as the *assembly instructions* that run every time you build a new object from that blueprint. `super().__init__()` means "before doing my own assembly steps, run the parent blueprint's assembly steps first."

## Key Takeaways

- PyTorch models are Python classes that inherit from `nn.Module`, always calling `super().__init__()` first.
- `*args`/`**kwargs` let functions accept flexible arguments — used everywhere in PyTorch's API.
- Training loops are `for` loops that unpack `(input, target)` tuples each iteration.
- `with` blocks guarantee setup/cleanup around a block of code — `torch.no_grad()` uses this pattern.
- `@decorator` syntax wraps a function with extra behavior — `@torch.no_grad()` is the most common one.
- PyTorch tensors extend the "operate on the whole array at once" idea from NumPy.

## What's Next

**Module 01 — What Is PyTorch, and Your First Tensors** introduces `torch.Tensor` itself: what it is, how it differs from a Python list or a NumPy array, and why it's the foundation everything else in this course is built on.

## Checklist

- [ ] I understand why `super().__init__()` is called inside a subclass's `__init__`
- [ ] I know the difference between `*args` and `**kwargs` and can read functions that use them
- [ ] I can read a `for x, y in something:` loop and know what's being unpacked
- [ ] I understand what a `with` block guarantees
- [ ] I can read a lambda function and a decorator
- [ ] I can read type hints in function signatures
- [ ] I understand the idea of operating on a whole array at once instead of looping in Python
